# PC-FNO3D v8 — corriger le DSO1

La v7 respecte **cinq objectifs sur six**. Seul le DSO1 (RMSE ≤ 0,5 m/s **à chaque niveau**)
échoue, et seulement en haut de la colonne :

| niveau | RMSE u | RMSE v | verdict |
|---|---|---|---|
| 100 → 500 m | 0,457 – 0,481 | 0,394 – 0,456 | conforme |
| 650 m | **0,564** | 0,408 | hors seuil |
| 800 m | **0,632** | 0,466 | hors seuil |
| 1000 m | **0,665** | **0,508** | hors seuil |

R² ≥ 0,991 partout : le modèle suit bien la variabilité. C'est l'erreur **absolue** qui
dépasse, là où le vent est le plus fort.

Ce notebook teste les corrections **de la moins chère à la plus chère**, et s'arrête dès
qu'une suffit :

| étape | coût | ce qu'elle teste |
|---|---|---|
| **0** | ~5 min | le paquet **reproduit-il** les chiffres v7 ? (sinon rien d'autre ne vaut) |
| **A** | ~15 min | moyenne de 8 passes sur la v7, **sans réentraîner** |
| **B → D** | ~1 h | affinage v8 : poids par niveau, sélection DSO, capteurs complets |

**Ce qu'on ne sait pas encore :** si ces corrections suffisent. Elles visent la cause
diagnostiquée, mais seul ce notebook peut le mesurer. Les verdicts s'écrivent dans
`aneo_fno3d_runs/<v7|v8>/eval_pcfno3d/*/test_summary.json` — un dossier à part, pour ne
**jamais** écraser les résultats du notebook v7 (`eval_best_dso_v7/`), qui servent de
référence à l'étape 0.

**Comment l'exécuter : Exécution → Tout exécuter.** Le notebook décide seul :
- il **s'arrête** si l'étape 0 montre que le paquet ne reproduit pas la v7, ou si une
  commande échoue — l'erreur est juste au-dessus de la cellule arrêtée ;
- il **saute l'entraînement** (B à D) si l'étape A respecte déjà les 5 DSO.

Durée : ~30 min si l'étape A suffit, ~2 h sinon (dont ~1 h 15 d'entraînement). Si Colab
coupe, relance « Tout exécuter » : chaque étape dont le résultat est déjà sur le Drive
se saute, et l'entraînement reprend où il en était. (`REFAIRE = True` dans la cellule
d'installation pour tout recalculer.)

Prérequis : exécution **GPU**, `pcfno3d_code.zip` à la racine du Drive
(`python scripts/package_for_colab.py` sur ton poste), et le cache v7 sur le Drive
(`aneo_cache_v7`, créé par le notebook v7).

## Installation

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import shutil, sys, zipfile
from pathlib import Path

ARCHIVE = Path("/content/drive/MyDrive/pcfno3d_code.zip")
CIBLE = Path("/content/PFE")
if not ARCHIVE.exists():
    raise FileNotFoundError("Dépose pcfno3d_code.zip à la racine de MyDrive "
                            "(sur ton poste : python scripts/package_for_colab.py)")
if CIBLE.exists():
    shutil.rmtree(CIBLE)
CIBLE.mkdir(parents=True)
zipfile.ZipFile(ARCHIVE).extractall(CIBLE)
%cd /content/PFE
!pip install -q -e . 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

# cache v7 : du Drive vers le disque local de Colab (lecture bien plus rapide)
CACHE_DRIVE, CACHE_LOCAL = Path("/content/drive/MyDrive/aneo_cache_v7"), Path("/content/aneo_cache_v7")
if not (CACHE_LOCAL / "index.csv").exists():
    if not (CACHE_DRIVE / "index.csv").exists():
        raise FileNotFoundError(f"{CACHE_DRIVE} introuvable : lance le BLOC 3 du notebook v7, "
                                "ou `python -m pcfno3d -c colab build-cache`")
    print("copie du cache v7 vers le disque local…")
    shutil.copytree(CACHE_DRIVE, CACHE_LOCAL, dirs_exist_ok=True)


def lancer(commande: str) -> None:
    """Exécute une commande en affichant sa sortie, et ARRÊTE le notebook si elle échoue.

    Une ligne `!commande` qui échoue ne stoppe pas « Tout exécuter » : les cellules
    suivantes partiraient d'un résultat absent. D'où ce garde-fou.
    """
    ip = get_ipython()
    ip.system(commande)
    code = ip.user_ns.get("_exit_code", 0)
    if code != 0:
        raise RuntimeError(f"échec (code {code}) de : {commande}\n"
                           "Le notebook s'arrête ici : lis l'erreur juste au-dessus.")


REFAIRE = False   # True pour tout recalculer, même ce qui est déjà sur le Drive


def deja_fait(resultat, apres=None) -> bool:
    """Vrai si `resultat` existe sur le Drive — et, si `apres` est donné, s'il est plus
    récent que lui (un verdict plus ancien que le modèle qu'il juge est périmé).
    Permet de relancer « Tout exécuter » après une coupure sans tout refaire."""
    resultat = Path(resultat)
    if REFAIRE or not resultat.exists():
        return False
    if apres is not None and Path(apres).exists() \
            and resultat.stat().st_mtime < Path(apres).stat().st_mtime:
        return False
    return True


RUNS = Path("/content/drive/MyDrive/aneo_fno3d_runs")
V7 = "/content/drive/MyDrive/aneo_fno3d_runs/v7/best_dso_v7.pt"
V8 = RUNS / "v8"
V8_NECESSAIRE = True      # devient False si l'étape A suffit à respecter les 5 DSO
print("prêt")

## Étape 0 — le paquet reproduit-il la v7 ?

Le paquet `pcfno3d` est un portage du notebook v7. **Il n'a jamais été comparé
numériquement au notebook sur les vraies données.** Cette étape le fait : elle réévalue
`best_dso_v7.pt` avec le paquet, sans aucune correction.

Il doit retrouver le BLOC 10 de la v7, à l'arrondi près — en particulier RMSE u = **0,665**
à 1000 m et RMSE global u = **0,542**. **Si ce n'est pas le cas, ne va pas plus loin** : le
portage a un écart, et toute comparaison v7/v8 serait fausse.

In [ ]:
if deja_fait(RUNS / "v7/eval_pcfno3d/best_dso_v7/test_summary.json"):
    print("étape 0 déjà faite : son résultat est sur le Drive, on le relit plus bas")
else:
    lancer(f"python -m pcfno3d -c colab evaluate --checkpoint {V7}")

In [ ]:
import json
import pandas as pd

# la référence : ce que le NOTEBOOK v7 a écrit (BLOC 10), jamais touché par le paquet
ref_csv = RUNS / "v7/eval_best_dso_v7/test_par_niveau_synoptique.csv"
if ref_csv.exists():
    ref = pd.read_csv(ref_csv).set_index("z_m")
else:       # repli : valeurs relevées dans la sortie du BLOC 10 de fno3d_v7.ipynb
    ref = pd.DataFrame({"rmse_u": {100: 0.458, 500: 0.481, 650: 0.564, 800: 0.632, 1000: 0.665},
                        "rmse_v": {100: 0.440, 500: 0.394, 650: 0.408, 800: 0.466, 1000: 0.508}})
# ce que le PAQUET vient de mesurer, dans son propre dossier
pkg = json.loads((RUNS / "v7/eval_pcfno3d/best_dso_v7/test_summary.json").read_text())
lv = {r["z_m"]: r for r in pkg["par_niveau_synoptique"]}
ecart_max = 0.0
for z in ref.index:
    if z not in lv or pd.isna(ref.loc[z, "rmse_u"]):
        continue
    du = abs(lv[z]["rmse_u"] - ref.loc[z, "rmse_u"])
    dv = abs(lv[z]["rmse_v"] - ref.loc[z, "rmse_v"])
    ecart_max = max(ecart_max, du, dv)
    print(f"{z:5d} m | u paquet {lv[z]['rmse_u']:.3f} / notebook {ref.loc[z, 'rmse_u']:.3f}"
          f" | v paquet {lv[z]['rmse_v']:.3f} / notebook {ref.loc[z, 'rmse_v']:.3f}")
print(f"\nécart maximal au notebook v7 : {ecart_max:.4f} m/s")
# 0,01 m/s de marge : les masques de capteurs sont tirés de la même façon, mais l'ordre
# des opérations en bf16 peut différer légèrement
PORTAGE_OK = ecart_max < 0.01
if not PORTAGE_OK:
    # arrêt volontaire : entraîner une v8 sur un paquet qui ne reproduit pas la v7 coûterait
    # une heure de GPU et donnerait une comparaison fausse
    raise RuntimeError(f"ÉCART de {ecart_max:.4f} m/s : le paquet ne reproduit pas la v7. "
                       "Le notebook s'arrête ici ; signale ces chiffres.")
print("PORTAGE VALIDÉ : le paquet reproduit la v7, on peut continuer")

## Étape A — moyenne de passes, sans réentraîner

La v7 est **plus précise avec 50 % des capteurs qu'avec 100 %** (RMSE max u 0,593 contre
0,705, table de robustesse du BLOC 10) : un seul passage avec tous les capteurs n'est pas
son meilleur régime. Moyenner 8 passes, chacune sur un sous-ensemble différent des
capteurs, réduit la variance — avec les caractéristiques géostrophiques **recalculées par
passe** (le BLOC 10c du notebook v7 les réutilisait, ce qui sortait chaque passe de la
distribution d'entraînement ; il avait été interrompu sans résultat).

À surveiller : 8 passes coûtent 8 inférences. La v7 fait 91 ms ; il faut rester sous
1 s (DSO4). L'évaluation le mesure.

In [ ]:
if deja_fait(RUNS / "v7/eval_pcfno3d/best_dso_v7_tta8/test_summary.json"):
    print("étape A déjà faite : son résultat est sur le Drive, on le relit plus bas")
else:
    lancer(f"python -m pcfno3d -c colab evaluate --checkpoint {V7} --tta 8 --tta-keep 0.7")

In [ ]:
# La moyenne de passes suffit-elle ? Il faut les CINQ DSO — y compris le DSO4 : 8 passes
# coûtent 8 inférences, et une correction qui casse le temps de réponse n'en est pas une.
sA = json.loads((RUNS / "v7/eval_pcfno3d/best_dso_v7_tta8/test_summary.json").read_text())
verdicts = {"DSO1": sA["dso1_rmse"], "DSO2": sA["dso2_conforme"], "DSO3": sA["dso3_conforme"],
            "DSO4": sA["dso4_conforme"], "DSO5": sA["dso5_conforme"]}
for nom, ok in verdicts.items():
    print(f"  {nom} : {'CONFORME' if ok else 'NON CONFORME'}")
print(f"  inférence : {sA['dso4_inference_s'] * 1000:.0f} ms pour 8 passes")
V8_NECESSAIRE = not all(verdicts.values())
print("\n" + ("La moyenne de passes ne suffit pas : on entraîne la v8 (étapes B à D)."
              if V8_NECESSAIRE else
              "LA MOYENNE DE PASSES SUFFIT : les 5 DSO sont respectés avec la v7, sans "
              "réentraîner. Les étapes B à D vont se sauter d'elles-mêmes ; il suffira de "
              "servir la v7 avec inference.tta = 8."))

## Étape B — poids par niveau, mesurés sur la **validation**

La loss moyenne sur tous les nœuds ne correspond pas au DSO1, qui est un **maximum** par
niveau. Ces poids donnent plus d'importance aux niveaux hors seuil : w = (RMSE/0,5)², bornés,
normalisés pour que l'échelle de la loss ne change pas.

Ils sont mesurés sur la **validation**, jamais sur le test : régler l'entraînement sur le
test reviendrait à mesurer sa propre copie (la configuration refuse un fichier marqué
`test`).

In [ ]:
if not V8_NECESSAIRE:
    print("étape B sautée : l'étape A respecte déjà les 5 DSO")
elif deja_fait(V8 / "uv_level_weight.json"):
    print("étape B déjà faite : les poids sont sur le Drive")
else:
    V8.mkdir(parents=True, exist_ok=True)
    # même normalisation que la v7 : le modèle affiné en dépend
    shutil.copy2(RUNS / "v7/norm_stats.json", V8 / "norm_stats.json")
    lancer(f"python -m pcfno3d -c colab dso-weights --checkpoint {V7} "
           f"--out {V8}/uv_level_weight.json")

## Étape C — affinage v8 (~1 h)

À partir de la v7, 25 époques, taux d'apprentissage 10× plus bas. Trois corrections :
poids par niveau (étape B), 15 % d'exemples avec **tous** les capteurs, et sélection du
meilleur modèle sur le **score DSO** de validation.

`lambda_grad` reste à **0,15**. Le BLOC 9b de la v7 avait affiné en le doublant (0,3), ce
qui privilégie la rugosité au détriment de la RMSE : son échec ne disait donc rien de
l'affinage lui-même. Et ici le modèle d'origine est le **premier candidat** : si aucune
époque ne fait mieux, c'est la v7 qui reste — l'affinage ne peut pas dégrader.

**Sauvegardes et reprise.** Chaque époque est enregistrée sur le disque local de Colab
(`/content/run_local_v8`, rapide), et recopiée sur le Drive **toutes les 5 époques**, à la
fin, et si tu arrêtes la cellule — écrire les ~7 Go de `last.pt` sur le Drive à chaque
époque coûterait ~75 s par époque. Si Colab coupe net, relance les deux cellules
d'installation puis cette cellule : l'entraînement reprend depuis la dernière copie du
Drive, en perdant **au plus 4 époques** (~10 min).

In [ ]:
if V8_NECESSAIRE:
    lancer("python -m pcfno3d -c colab_v8 train")
else:
    print("étape C sautée : l'étape A respecte déjà les 5 DSO")

## Étape D — verdict v8, avec et sans moyenne de passes

In [ ]:
if not V8_NECESSAIRE:
    print("étape D sautée : l'étape A respecte déjà les 5 DSO")
else:
    V8_BEST = V8 / "best.pt"
    # un verdict plus ancien que best.pt juge un autre modele : on le refait
    for suffixe, options in (("", ""), ("_tta8", " --tta 8 --tta-keep 0.7")):
        if deja_fait(V8 / f"eval_pcfno3d/best{suffixe}/test_summary.json", apres=V8_BEST):
            print(f"évaluation v8{suffixe or ' (une passe)'} déjà faite pour ce modèle")
        else:
            lancer(f"python -m pcfno3d -c colab_v8 evaluate --checkpoint {V8_BEST}{options}")

## Bilan : v7 / v7 + moyenne / v8 / v8 + moyenne

In [ ]:
import pandas as pd

candidats = {
    "v7":            RUNS / "v7/eval_pcfno3d/best_dso_v7/test_summary.json",
    "v7 + 8 passes": RUNS / "v7/eval_pcfno3d/best_dso_v7_tta8/test_summary.json",
    "v8":            RUNS / "v8/eval_pcfno3d/best/test_summary.json",
    "v8 + 8 passes": RUNS / "v8/eval_pcfno3d/best_tta8/test_summary.json",
}
lignes = []
for nom, p in candidats.items():
    if not p.exists():
        continue
    s = json.loads(p.read_text())
    ev = [r for r in s["par_niveau_synoptique"] if r["couverture"] >= 0.05]
    lignes.append({
        "modèle": nom,
        "RMSE max u": max(r["rmse_u"] for r in ev),
        "RMSE max v": max(r["rmse_v"] for r in ev),
        "DSO1": s["dso1_rmse"], "DSO1 global": s["dso1_rmse_global_conforme"],
        "DSO2": s["dso2_conforme"], "DSO3": s["dso3_conforme"],
        "DSO4 (ms)": round(s["dso4_inference_s"] * 1000),
        "DSO4": s["dso4_conforme"], "DSO5": s["dso5_conforme"],
    })
bilan = pd.DataFrame(lignes).set_index("modèle")
print(bilan.to_string())
tout = bilan[["DSO1", "DSO2", "DSO3", "DSO4", "DSO5"]].all(axis=1)
print("\nmodèles qui respectent les 5 DSO :", list(tout[tout].index) or "aucun")

**Lire le bilan honnêtement.**
- Une ligne qui respecte les cinq DSO est un candidat ; il faut alors la servir avec la
  même configuration (`inference.tta: 8` si elle passe par la moyenne).
- Si **aucune** ne passe, les niveaux 650–1000 m sont limités par l'information plutôt
  que par l'objectif d'entraînement : les capteurs sont à 25–30 m, et le haut de la
  colonne n'est relié aux mesures que par un vent géostrophique **uniforme** sur tout le
  domaine (un seul plan ajusté). La prochaine piste serait alors un estimateur
  géostrophique **spatialement variable**, ce qui change l'architecture d'entrée et
  demande un entraînement complet.
- Le critère OMM — RMSE ≤ max(0,5 ; 10 % de la vitesse moyenne) — est déjà **conforme**
  en v7 : en relatif, l'erreur en altitude est faible. Le seuil absolu de 0,5 m/s est le
  plus dur précisément là où le vent est le plus fort. Si la v8 ne suffit pas, c'est un
  point à discuter avec Tukhnanutha avant de lancer plus de calcul.